In [3]:
!pip install -q sentence-transformers faiss-cpu
!rm -rf repo && git clone --depth 1 https://github.com/shivarv/GDG-Hacakthon-nov21.git repo
!ls -lh repo/dataset/data/kb_train.parquet

Cloning into 'repo'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 148 (delta 2), reused 108 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 34.97 MiB | 20.08 MiB/s, done.
Resolving deltas: 100% (2/2), done.
-rw-r--r-- 1 root root 30M Nov 21 19:17 repo/dataset/data/kb_train.parquet


In [2]:
import pandas as pd, numpy as np, faiss, torch
from sentence_transformers import SentenceTransformer

print("GPU:", torch.cuda.is_available())

kb = pd.read_parquet("repo/dataset/data/kb_train.parquet")
kb = kb[kb["has_answer"] == True].reset_index(drop=True)
print("Rows with answers:", len(kb), kb["split"].value_counts().to_dict())

emb = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
emb.max_seq_length = 256
vecs = emb.encode(("passage: " + kb["text"].fillna("")).tolist(), batch_size=128,
                  normalize_embeddings=True, show_progress_bar=True).astype("float32")

def build(mask, name):
    idx = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs[mask])
    faiss.write_index(idx, f"{name}.index")
    kb.loc[mask, ["ticket_id", "split"]].reset_index(drop=True).to_parquet(f"{name}_ids.parquet", index=False)
    print(name, "->", idx.ntotal, "vectors")

build((kb["split"] == "base").values, "kb_base")
build(np.ones(len(kb), dtype=bool), "kb_all")

GPU: True
Rows with answers: 33412 {'base': 28959, 'dev': 4453}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/262 [00:00<?, ?it/s]

kb_base -> 28959 vectors
kb_all -> 33412 vectors


In [4]:
from google.colab import files
for f in ["kb_base.index", "kb_base_ids.parquet", "kb_all.index", "kb_all_ids.parquet"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>